<a href="https://colab.research.google.com/github/aiportfoliorhea/ai-portfolio/blob/main/loan_bot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn chromadb langchain pypdf sentence-transformers langchain-community langchain-text-splitters langchain-chroma anthropic

In [ ]:
import chromadb
import langchain
import fastapi

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
from langchain_chroma import Chroma
import chromadb
vector_store = Chroma(
    collection_name="loanbot",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

print("Vector store ready")

In [ ]:
from langchain_community.document_loaders import TextLoader
with open("jpm-10K.txt", "r", encoding="latin-1") as f:
    text = f.read(100000)

with open("jpm-10K-small.txt", "w") as f:
    f.write(text)

loader = TextLoader("jpm-10K-small.txt", encoding="latin-1")
docs = loader.load()

print(f"Loaded {len(docs)} document(s)")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
chunks = splitter.split_documents(docs)

print(f"Split into {len(chunks)} chunks")

In [ ]:
# @title
for i in range(len(chunks)):
    print(f"Chunk {i}: {chunks[i]}")

In [ ]:
import shutil
shutil.rmtree("./chroma_db", ignore_errors=True)

In [ ]:
vector_store.add_documents(chunks)
print("Chunks stored in vector store")

In [ ]:
retriever = vector_store.as_retriever()
results = retriever.invoke("What is JPMorgan's total loan portfolio size?")
print(results)

In [ ]:
for doc in results:
    print(doc.page_content)
    print("---")

In [ ]:
import anthropic
import os
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"]  = userdata.get('anthropic_key')
print("fetched Anthropic key succesfully")

In [ ]:
from anthropic import Anthropic

client = Anthropic()

def ask_loanbot(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1000,
        messages=[
            {
                "role": "user",
                "content": f"""Act as a legal loan document assistant. Answer the question using only the context provided.
If the answer is not in the context, say "I don't have that information in the document. Don't make up any anwers, strictly stick to the context given to you."

Context:
{context}

Question: {question}"""
            }
        ]
    )

    return message.content[0].text

print(ask_loanbot("What is JPMorgan's total loan portfolio size?"))
print("-----------------------------------")
print(ask_loanbot("What is the weather in New York?"))
